In [ ]:
import pandas as pd
from datetime import datetime,date
from glob import glob
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib


# 各社有車の稼働時間のまとめ

### 目的
## 社員が予約した内容から社有車の稼働時間を算出するための元データとしてまとめる
## 一つのデータフレームにまとめ、重複したデータは削除する。
## 生のデータフレームを返す(return)

csv_file_path = '../data/'
csv_file_pattern = csv_file_path + 'ご予約リスト_*.csv'


def load_and_combine_reservations():
    files = glob(csv_file_pattern)

    row_data = []

    for file in files:
        df = pd.read_csv(file,encoding = 'shift_jis')
        row_data.append(df)

    df = pd.concat(row_data,ignore_index=True)
    df = df.drop_duplicates()

    df[['予約日','稼働時間']] = df['予約日時'].str.split(' ',n=1,expand=True)
    df[['利用開始時刻','利用終了時刻']] = df['稼働時間'].str.split('\r\n~',expand=True)
    df['利用開始時刻'] = df['利用開始時刻'].str.replace('：',':')
    df['利用終了時刻'] = df['利用終了時刻'].str.replace('：',':')
    df['予約日'] = pd.to_datetime(df['予約日'],errors='coerce')
    df['年度'] = df['予約日'].dt.year
    df['月度'] = df['予約日'].dt.month
    df['利用開始日時_str'] = df['予約日'].dt.strftime('%Y-%m-%d') +' '+df['利用開始時刻']
    df['利用終了日時_str'] = df['予約日'].dt.strftime('%Y-%m-%d') +' '+df['利用終了時刻']
    df['利用開始日時'] = pd.to_datetime(df['利用開始日時_str'],errors='coerce')
    df['利用終了日時'] = pd.to_datetime(df['利用終了日時_str'],errors='coerce')

    df = df.drop(columns=['予約日時','利用開始日時_str','利用終了日時_str','予約日','Unnamed: 11']
                 , errors='ignore')


    df['稼働時間'] = (df['利用終了日時'] - df['利用開始日時'])
    df['稼働時間_hour'] = df['稼働時間'].dt.total_seconds()/3600
    return df



TypeError: datetime64 type does not support sum operations